## **Ejercicio:**

En pythpn contruir un grid 15x15 con 5 obstaculos, un inicio y un final usando politicas de direccion y metodo de valores para trazar el mejor camino entre inicio y fin. mostrar con RL (reinforcement), mostrar las iteraciones

### Librerias

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import clear_output
from matplotlib.colors import ListedColormap

### Configuracion de gridworld

In [ ]:
FILAS = 15
COLUMNAS = 15

# Coordenadas en formato: (fila, columna)
INICIO = (14, 0)
FINAL = (0, 14)

# Cinco obstáculos
OBSTACULOS = {
    (3, 5),
    (5, 10),
    (7, 7),
    (10, 4),
    (11, 11)
}

# Acciones disponibles:
# nombre: (cambio_fila, cambio_columna, símbolo)
ACCIONES = {
    "arriba":    (-1, 0, "↑"),
    "abajo":     (1, 0, "↓"),
    "izquierda": (0, -1, "←"),
    "derecha":   (0, 1, "→")
}

# Parámetros del aprendizaje por refuerzo
RECOMPENSA_MOVIMIENTO = -1.0
RECOMPENSA_FINAL = 100.0

# Factor de descuento
GAMMA = 0.95

# Condición de convergencia
TOLERANCIA = 1e-8

# Tiempo entre cada iteración de la animación
PAUSA = 0.25

### Funcion para mover el agente

In [ ]:
def mover(estado, accion):
    """
    Intenta mover al agente en la dirección indicada.

    Si el movimiento sale del tablero o llega a un obstáculo,
    el agente permanece en la misma posición.
    """

    fila, columna = estado

    cambio_fila, cambio_columna, _ = ACCIONES[accion]

    nueva_fila = fila + cambio_fila
    nueva_columna = columna + cambio_columna

    siguiente_estado = (nueva_fila, nueva_columna)

    # Verificar si sale de la cuadrícula
    fuera_del_grid = not (
        0 <= nueva_fila < FILAS and
        0 <= nueva_columna < COLUMNAS
    )

    # Si sale del grid o encuentra un obstáculo, no se mueve
    if fuera_del_grid or siguiente_estado in OBSTACULOS:
        return estado

    return siguiente_estado

### Calcular la politica direccional

In [ ]:
def crear_politica(valores):
    """
    Obtiene la mejor dirección para cada estado utilizando
    los valores calculados mediante la ecuación de Bellman.
    """

    politica = {}

    for fila in range(FILAS):
        for columna in range(COLUMNAS):

            estado = (fila, columna)

            # No se calcula política para obstáculos ni para el estado final
            if estado in OBSTACULOS or estado == FINAL:
                continue

            mejor_accion = None
            mejor_valor = float("-inf")

            for accion in ACCIONES:

                siguiente_estado = mover(estado, accion)

                # Recompensa obtenida al ejecutar la acción
                if siguiente_estado == FINAL:
                    recompensa = RECOMPENSA_FINAL
                else:
                    recompensa = RECOMPENSA_MOVIMIENTO

                # Ecuación de Bellman
                valor_accion = (
                    recompensa
                    + GAMMA * valores[siguiente_estado]
                )

                if valor_accion > mejor_valor:
                    mejor_valor = valor_accion
                    mejor_accion = accion

            politica[estado] = mejor_accion

    return politica


### Iteracion de valores

In [ ]:
def iteracion_de_valores():
    """
    Ejecuta el algoritmo de iteración de valores.

    Retorna:
    - Matriz final de valores.
    - Política óptima.
    - Historial de las matrices de valores.
    - Cambios máximos de cada iteración.
    """

    # Inicialmente todos los estados tienen valor 0
    valores = np.zeros((FILAS, COLUMNAS), dtype=float)

    historial = [valores.copy()]
    cambios = []

    numero_iteracion = 0

    while True:

        numero_iteracion += 1

        valores_anteriores = valores.copy()

        for fila in range(FILAS):
            for columna in range(COLUMNAS):

                estado = (fila, columna)

                # Los obstáculos y el estado final no se actualizan
                if estado in OBSTACULOS or estado == FINAL:
                    continue

                valores_acciones = []

                for accion in ACCIONES:

                    siguiente_estado = mover(estado, accion)

                    if siguiente_estado == FINAL:
                        recompensa = RECOMPENSA_FINAL
                    else:
                        recompensa = RECOMPENSA_MOVIMIENTO

                    # Ecuación de Bellman
                    valor_accion = (
                        recompensa
                        + GAMMA * valores_anteriores[siguiente_estado]
                    )

                    valores_acciones.append(valor_accion)

                # Se escoge el valor de la mejor acción
                valores[estado] = max(valores_acciones)

        # Cambio máximo entre la matriz anterior y la nueva
        cambio_maximo = np.max(
            np.abs(valores - valores_anteriores)
        )

        cambios.append(cambio_maximo)
        historial.append(valores.copy())

        print(
            f"Iteración {numero_iteracion:02d} | "
            f"Cambio máximo: {cambio_maximo:.8f}"
        )

        # Cuando el cambio es muy pequeño, el algoritmo converge
        if cambio_maximo < TOLERANCIA:
            break

    politica_optima = crear_politica(valores)

    return valores, politica_optima, historial, cambios


### Encontrar el mejor camino

In [ ]:
def obtener_mejor_camino(politica):
    """
    Sigue la política desde el inicio hasta el final.
    """

    estado_actual = INICIO

    camino = [estado_actual]
    estados_visitados = {estado_actual}

    limite_movimientos = FILAS * COLUMNAS

    for _ in range(limite_movimientos):

        if estado_actual == FINAL:
            return camino

        accion = politica[estado_actual]
        siguiente_estado = mover(estado_actual, accion)

        # Detectar posibles ciclos
        if siguiente_estado in estados_visitados:
            raise RuntimeError(
                "La política produjo un ciclo antes de llegar al final."
            )

        camino.append(siguiente_estado)
        estados_visitados.add(siguiente_estado)

        estado_actual = siguiente_estado

    raise RuntimeError(
        "No se encontró el estado final dentro del límite."
    )

### Mostrar una iteracion

In [ ]:
def mostrar_iteracion(valores, numero_iteracion, cambio=None):
    """
    Muestra los valores y la política direccional
    correspondiente a una iteración.
    """

    # Copiar los valores para representar los obstáculos
    matriz_visual = valores.copy()

    for obstaculo in OBSTACULOS:
        matriz_visual[obstaculo] = np.nan

    politica = crear_politica(valores)

    fig, ax = plt.subplots(figsize=(11, 9))

    imagen = ax.imshow(
        matriz_visual,
        cmap="viridis",
        origin="upper"
    )

    # Dibujar cada celda
    for fila in range(FILAS):
        for columna in range(COLUMNAS):

            estado = (fila, columna)

            # Obstáculos
            if estado in OBSTACULOS:

                rectangulo = plt.Rectangle(
                    (columna - 0.5, fila - 0.5),
                    1,
                    1,
                    facecolor="black"
                )

                ax.add_patch(rectangulo)

                ax.text(
                    columna,
                    fila,
                    "X",
                    ha="center",
                    va="center",
                    color="white",
                    fontweight="bold"
                )

            # Estado final
            elif estado == FINAL:

                ax.text(
                    columna,
                    fila,
                    "F",
                    ha="center",
                    va="center",
                    color="red",
                    fontsize=15,
                    fontweight="bold"
                )

            # Estado inicial
            elif estado == INICIO:

                ax.text(
                    columna,
                    fila,
                    "I",
                    ha="center",
                    va="center",
                    color="lime",
                    fontsize=15,
                    fontweight="bold"
                )

            # Política direccional
            else:

                accion = politica[estado]
                simbolo = ACCIONES[accion][2]

                ax.text(
                    columna,
                    fila,
                    simbolo,
                    ha="center",
                    va="center",
                    color="white",
                    fontsize=9,
                    fontweight="bold"
                )

    # Líneas de la cuadrícula
    ax.set_xticks(
        np.arange(-0.5, COLUMNAS, 1),
        minor=True
    )

    ax.set_yticks(
        np.arange(-0.5, FILAS, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="gray",
        linewidth=0.5
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    ax.set_xticks(range(COLUMNAS))
    ax.set_yticks(range(FILAS))

    titulo = f"Iteración de valores: {numero_iteracion}"

    if cambio is not None:
        titulo += f" | Cambio máximo: {cambio:.6f}"

    ax.set_title(titulo, fontsize=14)

    ax.set_xlabel("Columnas")
    ax.set_ylabel("Filas")

    plt.colorbar(
        imagen,
        ax=ax,
        label="Valor del estado V(s)"
    )

    plt.tight_layout()
    plt.show()


### Animacion de las iteraciones

In [ ]:
def animar_iteraciones(historial, cambios):
    """
    Muestra cada iteración como una animación en Google Colab.
    """

    for numero, valores in enumerate(historial):

        clear_output(wait=True)

        if numero == 0:
            mostrar_iteracion(
                valores,
                numero_iteracion=0
            )
        else:
            mostrar_iteracion(
                valores,
                numero_iteracion=numero,
                cambio=cambios[numero - 1]
            )

        time.sleep(PAUSA)


### Mostrar el camino optimo final

In [ ]:
def mostrar_camino_final(politica, camino):
    """
    Muestra la política óptima y resalta el mejor camino.
    """

    # 0 = libre
    # 1 = obstáculo
    # 2 = camino
    # 3 = inicio
    # 4 = final

    tablero = np.zeros((FILAS, COLUMNAS), dtype=int)

    for obstaculo in OBSTACULOS:
        tablero[obstaculo] = 1

    for estado in camino[1:-1]:
        tablero[estado] = 2

    tablero[INICIO] = 3
    tablero[FINAL] = 4

    colores = ListedColormap([
        "white",     # Espacio libre
        "black",     # Obstáculo
        "#74b9ff",   # Camino óptimo
        "#2ecc71",   # Inicio
        "#e74c3c"    # Final
    ])

    fig, ax = plt.subplots(figsize=(11, 9))

    ax.imshow(
        tablero,
        cmap=colores,
        vmin=0,
        vmax=4
    )

    # Dibujar política
    for fila in range(FILAS):
        for columna in range(COLUMNAS):

            estado = (fila, columna)

            if estado in politica:

                accion = politica[estado]
                simbolo = ACCIONES[accion][2]

                ax.text(
                    columna,
                    fila,
                    simbolo,
                    ha="center",
                    va="center",
                    fontsize=9
                )

    # Marcar inicio y final
    ax.text(
        INICIO[1],
        INICIO[0],
        "I",
        ha="center",
        va="center",
        fontsize=15,
        fontweight="bold"
    )

    ax.text(
        FINAL[1],
        FINAL[0],
        "F",
        ha="center",
        va="center",
        fontsize=15,
        fontweight="bold"
    )

    # Líneas del tablero
    ax.set_xticks(
        np.arange(-0.5, COLUMNAS, 1),
        minor=True
    )

    ax.set_yticks(
        np.arange(-0.5, FILAS, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="gray",
        linewidth=0.5
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    ax.set_xticks(range(COLUMNAS))
    ax.set_yticks(range(FILAS))

    ax.set_xlabel("Columnas")
    ax.set_ylabel("Filas")

    ax.set_title(
        "Camino óptimo mediante iteración de valores",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()


### Ejecucion del programa

In [ ]:
# Ejecutar iteración de valores
valores_optimos, politica_optima, historial, cambios = (
    iteracion_de_valores()
)

# Obtener el mejor camino
mejor_camino = obtener_mejor_camino(politica_optima)

# Mostrar animación de las iteraciones
animar_iteraciones(historial, cambios)

# Limpiar la animación anterior
clear_output(wait=True)

# Mostrar resultados en texto
print("=" * 60)
print("RESULTADO DEL APRENDIZAJE POR REFUERZO")
print("=" * 60)

print(f"Dimensiones del grid: {FILAS} x {COLUMNAS}")
print(f"Estado inicial: {INICIO}")
print(f"Estado final: {FINAL}")
print(f"Obstáculos: {sorted(OBSTACULOS)}")
print(f"Cantidad de iteraciones: {len(cambios)}")
print(f"Cantidad de movimientos: {len(mejor_camino) - 1}")
print(f"Valor del estado inicial: {valores_optimos[INICIO]:.4f}")

print("\nMejor camino encontrado:")

for numero, estado in enumerate(mejor_camino):
    print(f"Paso {numero:02d}: {estado}")

# Mostrar el resultado final
mostrar_camino_final(
    politica_optima,
    mejor_camino
)

**En que casos funciona mejor el metodo de politica?**

El método de políticas funciona mejor cuando el entorno es pequeño o mediano y es fácil evaluar una estrategia completa. Primero analiza qué tan buena es la política actual y luego cambia las decisiones que pueden mejorarse, por lo que normalmente necesita pocas mejoras para encontrar la política óptima.


**En que casos funcionan mejor el metodo de valores?**
El método de valores funciona mejor cuando hay muchos estados y se quiere encontrar directamente la mejor acción para cada posición. Actualiza poco a poco el valor de los estados hasta encontrar el mejor camino, por lo que suele ser más sencillo de programar y visualizar en problemas como una cuadrícula.